Lấy tất cả bài viết trong ngày

In [ ]:
import requests
from bs4 import BeautifulSoup
from datetime import date
import time
import random
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter
from datetime import datetime, timedelta

# 1. Cấu hình

In [ ]:

headers = {"User-Agent": "Mozilla/5.0"}
ngay = (date.today() - timedelta(days=0)).strftime("%Y-%m-%d")
all_links = []
page = 1

# 2. Lấy danh sách link

In [3]:
print(f"=== LINK NGÀY {ngay} ===")
while True:
    if page == 1:
        url = f"https://dantri.com.vn/thoi-su/from/{ngay}/to/{ngay}.htm"
    else:
        url = f"https://dantri.com.vn/thoi-su/from/{ngay}/to/{ngay}/trang-{page}.htm"

    res = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")

    articles = soup.find_all(
        "article",
        class_="article-item",
        attrs={"data-content-name": f"category-timeline_page_{page}"}
    )

    if not articles:
        break

    for art in articles:
        link = art.get("data-content-target")
        if link:
            if link.startswith("/"):
                link = "https://dantri.com.vn" + link
                all_links.append(link)
    page += 1
print("\nTỔNG LINK:", len(all_links))
for l in all_links:
    print(l)


=== LINK NGÀY 2026-01-31 ===

TỔNG LINK: 45
https://dantri.com.vn/thoi-su/khong-khi-lanh-lan-rong-mien-bac-co-noi-duoi-8-do-c-20260131190104160.htm
https://dantri.com.vn/thoi-su/giai-cuu-tai-xe-mac-ket-trong-xe-tai-lat-nghieng-o-tuyen-quang-20260131225203989.htm
https://dantri.com.vn/thoi-su/chu-tich-quoc-hoi-ra-soat-ky-ve-nhan-su-trong-cong-tac-bau-cu-20260131205401303.htm
https://dantri.com.vn/thoi-su/thu-tuong-ap-dung-quy-dinh-cho-phep-trien-khai-nhanh-duong-sat-toc-do-cao-20260131215439218.htm
https://dantri.com.vn/du-lich/da-lat-khep-lai-le-hoi-hoa-mai-anh-dao-xuan-2026-20260131204953903.htm
https://dantri.com.vn/thoi-su/tong-thong-my-donald-trump-chuc-mung-tong-bi-thu-to-lam-20260131210227302.htm
https://dantri.com.vn/thoi-su/canh-sat-lam-dong-dung-xe-dac-chung-mo-duong-dua-nguoi-dan-di-cap-cuu-20260131174209035.htm
https://dantri.com.vn/thoi-su/tong-bi-thu-to-lam-sap-tham-campuchia-20260131191850822.htm
https://dantri.com.vn/thoi-su/thuc-hu-thong-tin-canh-sat-giao-thong-vao-dam-

# 3. Hàm crawl

In [4]:
def crawl_article(url):
    res = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")

    # ===== TITLE =====
    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else ""

    # ===== BODY =====
    paragraphs = []
    for p in soup.find_all("p"):
        text = p.get_text(strip=True)
        if text:
            paragraphs.append(text)

    body = "\n".join(paragraphs)

    return {
        "url": url,
        "title": title,
        "content": body
    }

In [5]:
print("\n=== TEST 1 BÀI ===")
test_url = all_links[2]
data = crawl_article(test_url)

print("URL:", data["url"])
print("TITLE:", data["title"])
print("CONTENT:")
print(data["content"])


=== TEST 1 BÀI ===
URL: https://dantri.com.vn/thoi-su/chu-tich-quoc-hoi-ra-soat-ky-ve-nhan-su-trong-cong-tac-bau-cu-20260131205401303.htm
TITLE: Chủ tịch Quốc hội: Rà soát kỹ về nhân sự trong công tác bầu cử
CONTENT:
Chiều 31/1, tại trụ sở Đảng ủy phường Bình Minh, Ủy viên Bộ Chính trị, Chủ tịch Quốc hội, Chủ tịch Hội đồng Bầu cử quốc gia Trần Thanh Mẫn đã dẫn đầu Đoàn giám sát, kiểm tra công tác bầu cử đại biểu Quốc hội khóa XVI và đại biểu HĐND các cấp nhiệm kỳ 2026-2031, làm việc với Ủy ban Bầu cử tỉnh Vĩnh Long và Ủy ban Bầu cử phường Bình Minh.
Cuộc họp nhằm đánh giá tiến độ và chỉ đạo các công tác chuẩn bị cho cuộc bầu cử sắp tới.
Chủ tịch Quốc hội Trần Thanh Mẫn phát biểu chỉ đạo (Ảnh: Bảo Kỳ).
Cần quan tâm công tác tuyên truyền khi Vĩnh Long có hơn 300.000 người Khmer
Báo cáo tại buổi làm việc, Phó Bí thư Tỉnh ủy, Chủ tịch UBND tỉnh, Chủ tịch Ủy ban Bầu cử tỉnh Vĩnh Long Trần Trí Quang cho biết, Vĩnh Long, sau khi hợp nhất từ ba tỉnh Bến Tre, Trà Vinh và Vĩnh Long, hiện có diệ

# 4. Cấu hình Splitter

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=[
        "\n\n",   # đoạn
        "\n",     # dòng
        ".", "!", "?", ":", ";", ","
    ]
)
dulieu = splitter.split_text(data["content"])
print("Số chunk:", len(dulieu))
for i, chunk in enumerate(dulieu):
    print(f"\n--- CHUNK {i} ---")
    print(chunk)


Số chunk: 7

--- CHUNK 0 ---
Chiều 31/1, tại trụ sở Đảng ủy phường Bình Minh, Ủy viên Bộ Chính trị, Chủ tịch Quốc hội, Chủ tịch Hội đồng Bầu cử quốc gia Trần Thanh Mẫn đã dẫn đầu Đoàn giám sát, kiểm tra công tác bầu cử đại biểu Quốc hội khóa XVI và đại biểu HĐND các cấp nhiệm kỳ 2026-2031, làm việc với Ủy ban Bầu cử tỉnh Vĩnh Long và Ủy ban Bầu cử phường Bình Minh.
Cuộc họp nhằm đánh giá tiến độ và chỉ đạo các công tác chuẩn bị cho cuộc bầu cử sắp tới.
Chủ tịch Quốc hội Trần Thanh Mẫn phát biểu chỉ đạo (Ảnh: Bảo Kỳ).
Cần quan tâm công tác tuyên truyền khi Vĩnh Long có hơn 300.000 người Khmer

--- CHUNK 1 ---
Chủ tịch Quốc hội Trần Thanh Mẫn phát biểu chỉ đạo (Ảnh: Bảo Kỳ).
Cần quan tâm công tác tuyên truyền khi Vĩnh Long có hơn 300.000 người Khmer
Báo cáo tại buổi làm việc, Phó Bí thư Tỉnh ủy, Chủ tịch UBND tỉnh, Chủ tịch Ủy ban Bầu cử tỉnh Vĩnh Long Trần Trí Quang cho biết, Vĩnh Long, sau khi hợp nhất từ ba tỉnh Bến Tre, Trà Vinh và Vĩnh Long, hiện có diện tích gần 6.300km2 và quy mô 

# 5. Xử lý hàng loạt

In [7]:
final_data = []
print("\n=== BẮT ĐẦU XỬ LÝ CHI TIẾT BÀI VIẾT ===")

for i, link in enumerate(all_links):
    print(f"[{i+1}/{len(all_links)}] Đang xử lý: {link}")

    article_data = crawl_article(link)

    if article_data and article_data["content"]:
        # Split text
        chunks = splitter.split_text(article_data["content"])

        # Lưu kết quả
        final_data.append({
            "url": link,
            "title": article_data["title"],
            "chunks": chunks
        })
        print(f"  -> OK: {len(chunks)} chunks")
    else:
        print("  -> Bỏ qua (không lấy được nội dung)")

    # Rate limiting
    sleep_time = random.uniform(1, 2)
    time.sleep(sleep_time)

print(f"\n=== HOÀN THÀNH ===")
print(f"Đã xử lý thành công: {len(final_data)} bài.")



=== BẮT ĐẦU XỬ LÝ CHI TIẾT BÀI VIẾT ===
[1/45] Đang xử lý: https://dantri.com.vn/thoi-su/khong-khi-lanh-lan-rong-mien-bac-co-noi-duoi-8-do-c-20260131190104160.htm
  -> Bỏ qua (không lấy được nội dung)
[2/45] Đang xử lý: https://dantri.com.vn/thoi-su/giai-cuu-tai-xe-mac-ket-trong-xe-tai-lat-nghieng-o-tuyen-quang-20260131225203989.htm
  -> OK: 3 chunks
[3/45] Đang xử lý: https://dantri.com.vn/thoi-su/chu-tich-quoc-hoi-ra-soat-ky-ve-nhan-su-trong-cong-tac-bau-cu-20260131205401303.htm
  -> OK: 7 chunks
[4/45] Đang xử lý: https://dantri.com.vn/thoi-su/thu-tuong-ap-dung-quy-dinh-cho-phep-trien-khai-nhanh-duong-sat-toc-do-cao-20260131215439218.htm
  -> OK: 5 chunks
[5/45] Đang xử lý: https://dantri.com.vn/du-lich/da-lat-khep-lai-le-hoi-hoa-mai-anh-dao-xuan-2026-20260131204953903.htm
  -> OK: 3 chunks
[6/45] Đang xử lý: https://dantri.com.vn/thoi-su/tong-thong-my-donald-trump-chuc-mung-tong-bi-thu-to-lam-20260131210227302.htm
  -> OK: 17 chunks
[7/45] Đang xử lý: https://dantri.com.vn/thoi-su

In [8]:
# 6. Lưu file JSON
output_file = "dulieu.json"
result = {
    "date": ngay,
    "articles": final_data
} 
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)
print(f"Đã lưu dữ liệu vào file: {output_file}")

Đã lưu dữ liệu vào file: dulieu.json
